# 7-5_2 IMDb から DataLoader を作る今の書き方

`7-5_IMDb_Dataset_DataLoader.ipynb` は、IMDb のレビューから tsv を作り、`torchtext.data.Field` と `Iterator` で感情分析用の DataLoader を作る。このファイルは、同じデータと前処理を、今の PyTorch の `Dataset` と `DataLoader` で書き直したものである。

単語 ID と学習済みベクトルの並べ方は `7-4-2_vectorize.ipynb` と同じ役割である。7-5 では英語の fastText を `TEXT.vocab.vectors` に載せ、文頭の `<cls>` と文末の `<eos>` を付けて長さ 256 に揃えている。

7-6 以降は、まだ 7-5 の処理をまとめた `utils/dataloader.py` を前提にしている。このファイルは 7-5 の読み替え用で、後ろの章はそのままにする。

# 7-5_2 学習目標

1.	テキスト形式のファイルから tsv を作り、`Dataset` と `DataLoader` で 7-5 と同じミニバッチを作れる

# 事前準備

- レビューは 7-5 と同じ `data/aclImdb/`。コンテナ起動時に展開してある
- 英語の fastText は `data/wiki-news-300d-1M.vec`
- tsv の作り方と前処理は 7-5 と同じ

# 1. IMDbデータセットをtsv形式に変換

この節は 7-5 の「1. IMDbデータセットをtsv形式に変換」と同じである。

torchtext には IMDb を読む関数がある。7-5 は、データセットが用意されていない場合でも同じ手順で読めるように、ファイルから tsv を作っている。

元データは http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz である。訓練とテストがそれぞれ 2.5 万件、合計 5 万件ある。ファイル名はデータ id と rating（1 から 10）で決まっている。rating は 10 がもっとも良い。4 以下が negative、7 以上が positive で、positive のラベルを `1`、negative のラベルを `0` にする。

In [ ]:
# tsv形式のファイルにします
import glob
import os
import io
import string


# 訓練データのtsvファイルを作成します

f = open('./data/IMDb_train.tsv', 'w')

path = './data/aclImdb/train/pos/'
for fname in glob.glob(os.path.join(path, '*.txt')):
    with io.open(fname, 'r', encoding="utf-8") as ff:
        text = ff.readline()

        # タブがあれば消しておきます
        text = text.replace('\t', " ")

        text = text+'\t'+'1'+'\t'+'\n'
        f.write(text)

path = './data/aclImdb/train/neg/'
for fname in glob.glob(os.path.join(path, '*.txt')):
    with io.open(fname, 'r', encoding="utf-8") as ff:
        text = ff.readline()

        # タブがあれば消しておきます
        text = text.replace('\t', " ")

        text = text+'\t'+'0'+'\t'+'\n'
        f.write(text)

f.close()

In [ ]:
# テストデータの作成

f = open('./data/IMDb_test.tsv', 'w')

path = './data/aclImdb/test/pos/'
for fname in glob.glob(os.path.join(path, '*.txt')):
    with io.open(fname, 'r', encoding="utf-8") as ff:
        text = ff.readline()

        # タブがあれば消しておきます
        text = text.replace('\t', " ")

        text = text+'\t'+'1'+'\t'+'\n'
        f.write(text)


path = './data/aclImdb/test/neg/'

for fname in glob.glob(os.path.join(path, '*.txt')):
    with io.open(fname, 'r', encoding="utf-8") as ff:
        text = ff.readline()

        # タブがあれば消しておきます
        text = text.replace('\t', " ")

        text = text+'\t'+'0'+'\t'+'\n'
        f.write(text)

f.close()

# 2. 前処理と単語分割の関数を定義

この節は 7-5 の「2. 前処理と単語分割の関数を定義」と同じである。英語のレビューなので、記号を整理したあとスペースで区切る。

In [ ]:
import string
import re

# 以下の記号はスペースに置き換えます（カンマ、ピリオドを除く）。
# punctuationとは日本語で句点という意味です
print("区切り文字：", string.punctuation)
# !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~

# 前処理


def preprocessing_text(text):
    # 改行コードを消去
    text = re.sub('<br />', '', text)

    # カンマ、ピリオド以外の記号をスペースに置換
    for p in string.punctuation:
        if (p == ".") or (p == ","):
            continue
        else:
            text = text.replace(p, " ")

    # ピリオドなどの前後にはスペースを入れておく
    text = text.replace(".", " . ")
    text = text.replace(",", " , ")
    return text

# 分かち書き（今回はデータが英語で、簡易的にスペースで区切る）


def tokenizer_punctuation(text):
    return text.strip().split()


# 前処理と分かち書きをまとめた関数を定義
def tokenizer_with_preprocessing(text):
    text = preprocessing_text(text)
    ret = tokenizer_punctuation(text)
    return ret


# 動作を確認します
print(tokenizer_with_preprocessing('I like cats.'))

# 文章データの読み込み

7-5 では、読み込んだ文章とラベルに対する処理を `Field` で定義し、`TabularDataset` で tsv を読んでいた。

```python
max_length = 256
TEXT = torchtext.data.Field(sequential=True, tokenize=tokenizer_with_preprocessing, use_vocab=True,
                            lower=True, include_lengths=True, batch_first=True, fix_length=max_length, init_token="<cls>", eos_token="<eos>")
LABEL = torchtext.data.Field(sequential=False, use_vocab=False)

train_val_ds, test_ds = torchtext.data.TabularDataset.splits(
    path='./data/', train='IMDb_train.tsv',
    test='IMDb_test.tsv', format='tsv',
    fields=[('Text', TEXT), ('Label', LABEL)])
```

`Field` の引数は、次の場所に分ける。

| 7-5 の引数 | このファイルでの置き場所 |
| --- | --- |
| `tokenize` | `Dataset` を作るときに `tokenizer_with_preprocessing` を呼ぶ |
| `lower=True` | 単語に分けたあと、各単語を小文字にする |
| `use_vocab=True` | 次の節で単語を ID にする。ラベルは 7-5 の `use_vocab=False` と同じく数値のまま持つ |
| `init_token="<cls>"` と `eos_token="<eos>"` | `DataLoader` に渡す関数で、文頭と文末に付ける |
| `include_lengths=True` | 同じ関数で、パディング前の長さも返す |
| `batch_first=True` と `fix_length=256` | 同じ関数で、長さ 256 の `[バッチ, 256]` に揃える |

`fix_length=256` は `<cls>` と `<eos>` を含んだ長さである。レビュー本文は 254 単語まで入れる。`sequential` は「文章は長さが可変、ラベルは 1 個」という意味だった。ここでは文章をリスト、ラベルを 1 個の値として持てば足りる。

7-5 が書く各行は、文章、タブ、ラベル、タブの順である。`fields` が 2 個なので、末尾の空の列は読まない。

In [ ]:
from torch.utils.data import Dataset


class TextDataset(Dataset):
    """tsv の1行を (単語リスト, ラベル) として持つ。7-5 の TabularDataset に相当する。"""

    def __init__(self, samples):
        self.samples = list(samples)

    @classmethod
    def from_tsv(cls, path, tokenizer):
        samples = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                text, label, *_rest = line.rstrip("\n").split("\t")
                tokens = tokenizer(text)
                # Field は tokenize のあとで lower=True を各単語にかける
                tokens = [token.lower() for token in tokens]
                samples.append((tokens, label))
        return cls(samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


train_val_ds = TextDataset.from_tsv("./data/IMDb_train.tsv", tokenizer_with_preprocessing)
test_ds = TextDataset.from_tsv("./data/IMDb_test.tsv", tokenizer_with_preprocessing)

# 動作確認。7-5 の vars(train_val_ds[0]) と同じ中身になる
print("訓練および検証のデータ数", len(train_val_ds))
print("1つ目の訓練および検証のデータ", {"Text": train_val_ds[0][0], "Label": train_val_ds[0][1]})

7-5 は `Dataset.split` で訓練データと検証データに分けている。

```python
import random

train_ds, val_ds = train_val_ds.split(
    split_ratio=0.8, random_state=random.seed(1234))
```

`random.seed(1234)` の戻り値は `None` である。torchtext はその直後の乱数状態で、添字を `random.sample` して並び替える。先頭の `round(0.8 * 件数)` 件が訓練、残りが検証になる。

In [ ]:
import random


def split_train_val(dataset, split_ratio=0.8, seed=1234):
    """7-5 の Dataset.split(split_ratio=0.8, random_state=random.seed(1234)) に相当する。"""
    random.seed(seed)
    order = random.sample(range(len(dataset)), len(dataset))
    train_len = int(round(split_ratio * len(dataset)))
    train_samples = [dataset[index] for index in order[:train_len]]
    val_samples = [dataset[index] for index in order[train_len:]]
    return TextDataset(train_samples), TextDataset(val_samples)


train_ds, val_ds = split_train_val(train_val_ds)

# 動作確認
print("訓練データの数", len(train_ds))
print("検証データの数", len(val_ds))
print("1つ目の訓練データ", {"Text": train_ds[0][0], "Label": train_ds[0][1]})

# ボキャブラリーを作成

7-5 は `torchtext.vocab.Vectors` で英語の fastText を読み、`TEXT.build_vocab(train_ds, vectors=english_fasttext_vectors, min_freq=10)` で `TEXT.vocab.vectors` を作っていた。

```python
from torchtext.vocab import Vectors

english_fasttext_vectors = Vectors(name='data/wiki-news-300d-1M.vec')
TEXT.build_vocab(train_ds, vectors=english_fasttext_vectors, min_freq=10)
```

ここでは gensim の `KeyedVectors` で同じファイルを読み、`stoi` の順番にベクトルを並べた行列を作る。語彙は訓練データだけから作る。`min_freq` 未満の単語は入れない。ID の並びは 7-5 に合わせる。`<unk>`、`<pad>`、`<cls>`、`<eos>` が先で、その後は出現回数が多い順、回数が同じなら文字コード順である。

`<unk>`、`<pad>`、`<cls>`、`<eos>`、学習済みファイルに無い単語は、7-5 と同じくゼロベクトルにする。

In [ ]:
from gensim.models import KeyedVectors

english_fasttext = KeyedVectors.load_word2vec_format(
    "data/wiki-news-300d-1M.vec", binary=False)

print("1単語を表現する次元数:", english_fasttext.vector_size)
print("単語数:", len(english_fasttext.vocab))

In [ ]:
from collections import Counter

import torch


def build_vocab(dataset, min_freq=10):
    """7-5 の TEXT.build_vocab(train_ds, min_freq=10) に相当する。"""
    freqs = Counter()
    for tokens, _label in dataset:
        freqs.update(tokens)

    specials = ["<unk>", "<pad>", "<cls>", "<eos>"]
    for token in specials:
        if token in freqs:
            del freqs[token]

    words = [word for word, freq in freqs.items() if freq >= min_freq]
    words.sort(key=lambda word: (-freqs[word], word))
    itos = specials + words
    stoi = {word: index for index, word in enumerate(itos)}
    return stoi, itos


def build_vector_table(keyed_vectors, stoi):
    """7-5 の TEXT.vocab.vectors に相当する行列を作る。"""
    table = torch.zeros(len(stoi), keyed_vectors.vector_size)
    for word, index in stoi.items():
        if word in keyed_vectors:
            table[index] = torch.tensor(keyed_vectors[word].copy())
    return table


stoi, itos = build_vocab(train_ds, min_freq=10)
vectors = build_vector_table(english_fasttext, stoi)

# 7-5 の実行例では torch.Size([17916, 300])
print(vectors.shape)
vectors

In [ ]:
# 単語を id に変換する辞書。7-5 の TEXT.vocab.stoi に相当する
# 辞書に無い単語は <unk> の 0 になる
stoi

# DataLoaderの作成

7-5 では `torchtext.data.Iterator` がミニバッチを作っていた。検証データとテストデータは `train=False, sort=False` なので、データセットの順のまま取り出していた。訓練データは `train=True` なので、取り出す順をシャッフルしていた。

```python
train_dl = torchtext.data.Iterator(train_ds, batch_size=24, train=True)

val_dl = torchtext.data.Iterator(
    val_ds, batch_size=24, train=False, sort=False)

test_dl = torchtext.data.Iterator(
    test_ds, batch_size=24, train=False, sort=False)

batch = next(iter(val_dl))
print(batch.Text)
print(batch.Label)
```

`batch.Text` は `(単語IDのテンソル, 各文の単語数)` だった。`include_lengths=True` のためである。`batch_first=True` と `fix_length=256` のため、テンソルの形は `[バッチ, 256]` になる。

長さ 256 には `<cls>` と `<eos>` が含まれる。本文は 254 単語までで、短い文は `<pad>` で埋める。単語数は `<cls>` と `<eos>` を含み、`<pad>` を含まない。

その整形を `collate_batch` に書く。`DataLoader` はバッチを取り出すときにこの関数を呼ぶ。

In [ ]:
from torch.utils.data import DataLoader

max_length = 256
content_length = max_length - 2  # <cls> と <eos> の分を除いた本文の長さ
pad_id = stoi["<pad>"]
unk_id = stoi["<unk>"]
cls_id = stoi["<cls>"]
eos_id = stoi["<eos>"]


def collate_batch(batch):
    """7-5 の include_lengths、batch_first、fix_length、init_token、eos_token に相当する。"""
    text_batch = []
    lengths = []
    labels = []

    for tokens, label in batch:
        ids = [stoi.get(token, unk_id) for token in tokens[:content_length]]
        ids = [cls_id] + ids + [eos_id]
        length = len(ids)
        ids += [pad_id] * (max_length - len(ids))

        text_batch.append(torch.tensor(ids, dtype=torch.long))
        lengths.append(length)
        labels.append(int(label))

    text = torch.stack(text_batch)
    lengths = torch.tensor(lengths, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.long)
    return text, lengths, labels


# train=True はシャッフルする、という意味だった
train_dl = DataLoader(train_ds, batch_size=24, shuffle=True, collate_fn=collate_batch)
val_dl = DataLoader(val_ds, batch_size=24, shuffle=False, collate_fn=collate_batch)
test_dl = DataLoader(test_ds, batch_size=24, shuffle=False, collate_fn=collate_batch)

# 動作確認。7-5 の batch.Text と batch.Label に相当する
text, lengths, labels = next(iter(val_dl))
print((text, lengths))
print(labels)

このように DataLoader は単語の id を格納している。分散表現は、ディープラーニングモデル側で id に応じて `vectors` から取る。

7-5 はここまでの処理を `utils/dataloader.py` にまとめ、7-6 以降はそこから読み込む。このファイルは 7-5 の読み替えで、7-6 以降はその `dataloader.py` のままである。

以上